In [21]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [22]:
class AgentState(TypedDict):
    num1: int
    num2: int
    num3: int
    num4: int
    op1: str
    op2: str
    final1: int
    final2: int

In [23]:
def adder1(state: AgentState) -> AgentState:
    """This function adds the first two numbers, from the state."""
    state['final1'] = state['num1'] + state['num2']
    
    return state

def sub1(state: AgentState) -> AgentState:
    """This function subtracts the first two numbers, from the state."""
    state['final1'] = state['num1'] - state['num2']
    
    return state

In [24]:
def adder2(state: AgentState) -> AgentState:
    """This function adds the last two numbers, from the state."""
    state['final2'] = state['num3'] + state['num4']
    
    return state

def sub2(state: AgentState) -> AgentState:
    """This function subtracts the last two numbers, from the state."""
    state['final2'] = state['num3'] - state['num4']
    
    return state

In [25]:
def conditional_node1(state: AgentState):
    """This node routes the graph to either adder or sub, according to the provided op1."""
    if state['op1'] == '+':
        return "addition_operation"
    
    elif state['op1'] == '-':
        return "subtraction_operation"
    
def conditional_node2(state: AgentState):
    """This node routes the graph to either adder or sub, according to the provided op2."""
    if state['op2'] == '+':
        return "addition_operation2"
    
    elif state['op2'] == '-':
        return "subtraction_operation2"

In [28]:
graph = StateGraph(AgentState)

graph.add_node("add_node", adder1)
graph.add_node("add_node2", adder2)

graph.add_node("subtract_node", sub1)
graph.add_node("subtract_node2", sub2)

graph.add_node("router", lambda state: state)
graph.add_node("router2", lambda state: state)


graph.add_conditional_edges(
    "router",
    conditional_node1,
    
    {
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"
    }
)

graph.add_conditional_edges(
    "router2",
    conditional_node2,
    
    {
        "addition_operation2": "add_node2",
        "subtraction_operation2": "subtract_node2"
    }
)

graph.add_edge(START, "router")

graph.add_edge("add_node", "router2")
graph.add_edge("subtract_node", "router2")

graph.add_edge("add_node2", END)
graph.add_edge("subtract_node2", END)


app = graph.compile()

In [29]:
app.invoke({
    'num1': 1,
    'num2': 2,
    'num3': 3,
    'num4': 4,
    'op1': '+',
    'op2': '-'
})

{'num1': 1,
 'num2': 2,
 'num3': 3,
 'num4': 4,
 'op1': '+',
 'op2': '-',
 'final1': 3,
 'final2': -1}